# Chapter 12: Evals, Observability & AI Safety

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abda-dc/ai-engineering-zero-to-hero/blob/main/book/chapters/part4-agents-prod/ch12-evals-observability-and-safety.ipynb)

**Part:** Part IV: Autonomous Agents, MCP & Production MLOps  
**Chapter Identifier:** `ch12`  

## Core Concepts Covered in this Chapter:
- **Ragas & DeepEval**
- **Prompt A/B Testing**
- **Hallucination Detection**
- **OpenTelemetry Traces/Logs/Metrics**
- **Cost & Latency Optimization**
- **GDPR/HIPAA Privacy**
- **AI Alignment/Fairness**
- **Product UX**

---


## 1. Continuous Evals & The RAG Triad (Ragas / DeepEval)

You cannot improve what you cannot measure. Modern AI evaluations test the RAG Triad: (1) Context Precision (did we retrieve relevant passages?), (2) Faithfulness (is the answer grounded in context?), and (3) Answer Relevance (did we address the user's question?).

```mermaid
graph TD
    Query[User Query] --> C_Prec[Context Precision]
    Context[Retrieved Context] --> C_Prec
    Context --> Faith[Faithfulness Eval]
    Answer[Generated Answer] --> Faith
    Query --> A_Rel[Answer Relevance]
    Answer --> A_Rel
```


In [ ]:
def evaluate_rag_triad(query: str, context: list, answer: str) -> dict:
    q_words = set(query.lower().split())
    ctx_words = set(" ".join(context).lower().split())
    ans_words = set(answer.lower().split())
    
    prec = len(q_words.intersection(ctx_words)) / max(len(q_words), 1)
    faith = len(ans_words.intersection(ctx_words)) / max(len(ans_words), 1)
    rel = len(q_words.intersection(ans_words)) / max(len(q_words), 1)
    
    return {
        "context_precision": round(min(prec, 1.0), 3),
        "faithfulness": round(min(faith * 1.2, 1.0), 3),
        "answer_relevance": round(min(rel, 1.0), 3)
    }

q = "What is the capital of France?"
ctx = ["Paris is the capital and largest city of France.", "The Seine flows through Paris."]
ans = "The capital of France is Paris."

scores = evaluate_rag_triad(q, ctx, ans)
print("RAG Triad Evaluation Metrics:", scores)

## 2. OpenTelemetry Tracing for Multi-Step AI Pipelines

Complex agent and RAG workflows involve nested function calls. OpenTelemetry distributed tracing models these interactions as Spans with timestamps, latency, token usage, and error events.


In [ ]:
import uuid

class OpenTelemetrySpanSimulator:
    def __init__(self, name: str, parent_id: str = None):
        self.span_id = str(uuid.uuid4())[:8]
        self.name = name
        self.parent_id = parent_id
        self.attributes = {}
        self.events = []

    def set_attribute(self, key: str, value):
        self.attributes[key] = value

    def add_event(self, name: str):
        self.events.append(name)

    def to_dict(self):
        return {
            "span_id": self.span_id,
            "parent_id": self.parent_id,
            "name": self.name,
            "attributes": self.attributes,
            "events": self.events
        }

root_span = OpenTelemetrySpanSimulator("agent_request")
root_span.set_attribute("user.id", "usr_123")
child_span = OpenTelemetrySpanSimulator("vector_search", parent_id=root_span.span_id)
child_span.set_attribute("retrieval.k", 5)
child_span.set_attribute("latency_ms", 12.4)

print("Trace Tree Sample:")
print("Root:", root_span.to_dict())
print("Child:", child_span.to_dict())

## 3. AI Privacy, GDPR/HIPAA Redaction & Bias Guardrails

Enterprise AI must comply with privacy mandates (GDPR, HIPAA, SOC 2). Personally Identifiable Information (PII) such as SSNs, credit cards, emails, and medical identifiers must be scrubbed before sending prompts to external models.


In [ ]:
import re

def redact_pii(text: str) -> str:
    redacted = re.sub(r'[a-zA-Z0-9_.+-]+@[a-zA-Z0-9-]+\.[a-zA-Z0-9-.]+', '[REDACTED_EMAIL]', text)
    redacted = re.sub(r'\b\d{3}-\d{2}-\d{4}\b', '[REDACTED_SSN]', redacted)
    redacted = re.sub(r'\b(?:\d[ -]*?){13,16}\b', '[REDACTED_CC]', redacted)
    return redacted

user_submission = "Patient Jane Doe (SSN: 987-65-4321, email: jane.doe@health.org) requests medical records."
print("Sanitized for GDPR/HIPAA Compliance:\n", redact_pii(user_submission))

---
## Hands-On Capstone Lab: Build a Unified Evaluation, Tracing & Privacy Guardrail Harness

### Objective:
Build a safety and observability wrapper that redacts incoming PII, logs an OpenTelemetry span, and executes automated faithfulness evaluation on the output.

### Instructions:
1. Sanitize prompt text with PII redaction.
2. Trace execution time and token metrics.
3. Calculate Faithfulness score of the generated text against context.


In [ ]:
import time, uuid

def run_observability_pipeline(user_query: str, retrieved_context: list, generated_answer: str) -> dict:
    t0 = time.time()
    clean_query = redact_pii(user_query)
    eval_metrics = evaluate_rag_triad(clean_query, retrieved_context, generated_answer)
    duration_ms = round((time.time() - t0) * 1000 + 15.0, 2)
    return {
        "sanitized_query": clean_query,
        "eval_scores": eval_metrics,
        "trace": {
            "trace_id": str(uuid.uuid4())[:12],
            "duration_ms": duration_ms,
            "safety_passed": eval_metrics["faithfulness"] >= 0.70
        }
    }

context_passages = ["Antigravity delivers autonomous coding capabilities with verified execution gates."]
query_input = "Contact support at admin@antigravity.ai regarding Antigravity capabilities."
answer_output = "Antigravity provides autonomous coding with verified gates."

obs_result = run_observability_pipeline(query_input, context_passages, answer_output)
print("Observability Output:")
print("  Clean Query:", obs_result["sanitized_query"])
print("  Evals:", obs_result["eval_scores"])
print("  Trace:", obs_result["trace"])

---
## Chapter Mastery Self-Check Quiz

Test your understanding of the concepts covered in this chapter.

### Question 1: In the RAG Triad evaluation framework, what is evaluated by 'Context Precision'?
- **A)** How fast the vector database queries run
- **B)** Whether all retrieved context passages are relevant to answering the user's query
- **C)** The font size of the output text
- **D)** Whether the user clicked like or dislike

<details>
<summary><b>View Answer & Explanation</b></summary>

**Correct Answer:** B) Whether all retrieved context passages are relevant to answering the user's query

**Explanation:** Context Precision measures the signal-to-noise ratio of the retrieval stage: whether the retrieved documents directly contribute to answering the query without noise.
</details>

### Question 2: What is the primary role of OpenTelemetry in generative AI observability?
- **A)** To generate synthetic training data
- **B)** To provide vendor-agnostic distributed tracing, capturing spans, latency, token costs, and tool calls across multi-step pipelines
- **C)** To fine-tune small language models
- **D)** To replace SQL databases

<details>
<summary><b>View Answer & Explanation</b></summary>

**Correct Answer:** B) To provide vendor-agnostic distributed tracing, capturing spans, latency, token costs, and tool calls across multi-step pipelines

**Explanation:** OpenTelemetry provides an open, vendor-neutral standard for traces, metrics, and logs, making multi-step LLM chains and agent tool invocations debuggable in production.
</details>

### Question 3: Under GDPR and HIPAA, what must occur before transmitting proprietary user records to third-party model APIs?
- **A)** The text must be translated into French
- **B)** Personally Identifiable Information (PII) and Protected Health Information (PHI) must be scrubbed, anonymized, or redacted
- **C)** The model must be restarted
- **D)** The temperature must be set to 2.0

<details>
<summary><b>View Answer & Explanation</b></summary>

**Correct Answer:** B) Personally Identifiable Information (PII) and Protected Health Information (PHI) must be scrubbed, anonymized, or redacted

**Explanation:** Regulatory frameworks mandate strict data protection; identifiable personal and medical information must be redacted or pseudonymized before external transmission.
</details>

### Question 4: What does a Hallucination Detection guardrail primarily check for?
- **A)** Spelling errors and grammar mistakes
- **B)** Factual assertions or claims made by the model that contradict or cannot be inferred from the reference context
- **C)** Slow internet connections
- **D)** Missing CSS styles

<details>
<summary><b>View Answer & Explanation</b></summary>

**Correct Answer:** B) Factual assertions or claims made by the model that contradict or cannot be inferred from the reference context

**Explanation:** Hallucination detectors test for unsubstantiated claims in the model's response that lack grounding in verified source evidence.
</details>
